# Installations

In [ ]:
# !pip install -q wandb

# Load libraries and setup

In [ ]:
from zzzs_util import *

import os
import gc
import ctypes
import joblib
import wandb
from tqdm.auto import tqdm

libc = ctypes.CDLL("libc.so.6")

import pytz
# import cudf
import pandas as pd
import numpy as np
import tensorflow as tf

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

# import torch
# from torch.utils.data import Dataset, DataLoader
from transformers import Trainer, TrainingArguments

from sklearn.preprocessing import RobustScaler, StandardScaler

# print(cudf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import Concatenate, LSTM, GRU
from tensorflow.keras.layers import Bidirectional, Multiply

np.random.seed(101)
tf.random.set_seed(101)

In [ ]:
COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"


In [ ]:
# # Setup user secrets for login
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# WANDB_KEY = user_secrets.get_secret("wandb")

# import wandb
# wandb.login(key=WANDB_KEY)
# wandb.init(project='kaggle-zzzs')

# LSTM model functions

In [ ]:
# # Define a function to generate sequences from the data
# def generate_sequences(data, targets, sequence_length=128):
#     X, y = [], []
#     for i in range(len(data) - sequence_length):
#         X.append(data[i: i + sequence_length])
#         y.append(targets[i + sequence_length])
#     return np.array(X), np.array(y)

In [ ]:
# # turn target inds into array
# target_guassian = np.zeros((len(X),2))
# for s,e in y:
#     st1,st2 = max(0,s-SIGMA//2),s+SIGMA//2+1
#     ed1,ed2 = e-SIGMA//2,min(len(X),e+SIGMA//2+1)
#     target_guassian[st1:st2,0] = self.gauss()[st1-(s-SIGMA//2):]
#     target_guassian[ed1:ed2,1] = self.gauss()[:SIGMA+1-((e+SIGMA//2+1)-ed2)]
#     gc.collect()
# y = target_guassian

In [ ]:
# def generate_sequences(data, targets, sequence_length=128):
#     X, y = [], []
#     # Placeholder value for no event
#     NO_EVENT = -1
    
#     for i in range(len(data) - sequence_length):
#         sequence = data[i: i + sequence_length]
#         X.append(sequence)
        
#         # Check if any onset or wakeup event exists within this sequence
#         label = NO_EVENT  # Default no event
#         for onset, wakeup in targets:
#             # Check if onset is within this sequence
#             if i <= onset < i + sequence_length:
#                 label = onset - i
#                 break
#             # Check if wakeup is within this sequence
#             elif i <= wakeup < i + sequence_length:
#                 label = wakeup - i
#                 break
        
#         y.append(label)
    
#     return np.array(X), np.array(y)

In [ ]:
# # Define the LSTM model with l2 regularization for weight decay
# def build_lstm_model(input_shape, weight_decay):
#     model = tf.keras.Sequential([
#         tf.keras.layers.LSTM(50, input_shape=input_shape, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(weight_decay)),
#         tf.keras.layers.LSTM(50, kernel_regularizer=tf.keras.regularizers.l2(weight_decay)),
#         tf.keras.layers.Dense(1, kernel_regularizer=tf.keras.regularizers.l2(weight_decay))
#     ])
#     optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
#     model.compile(optimizer=optimizer, loss='mse')
#     return model


In [ ]:
# Custom learning rate scheduler with warmup
def lr_schedule(epoch, lr, warmup_epochs=5, warmup_lr=1e-5):
    if epoch < warmup_epochs:
        return (lr - warmup_lr) / warmup_epochs * epoch + warmup_lr
    else:
        return lr

In [ ]:
def data_cleaning(data):
    
    # Data cleaning
    data = data.fillna(0)
    data = data.astype(float)
    data = data.to_numpy()
    
    return data

In [ ]:
def compute_new_lr(epoch, initial_lr=1e-4, final_lr=1e-6, total_epochs=10):
    return initial_lr - (initial_lr - final_lr) * (epoch / total_epochs)

In [ ]:
def generate_target_array(data_length, targets):
    """
    Generates a binary target numpy array based on the provided targets.
    
    Parameters:
    - data_length (int): Length of the data series.
    - targets (list of tuples): Each tuple is a pair of start and end indices for sleep events.
    
    Returns:
    - numpy array: Binary target array where 1 indicates awake and 0 indicates sleep.
    """
    y = np.ones(data_length, dtype=int)  # Start with an array of ones (awake)

    for start, end in targets:
        y[int(start):int(end)+1] = 0  # Set the indices corresponding to sleep events to 0

    return y.reshape(-1, 1)

In [ ]:
# # Define the number of data points in a 48-hour period
# POINTS_IN_48H = int(48 * 60 * 60 / 5)

# def split_into_chunks(X, y):
#     # Split X into chunks
#     num_chunks = len(X) // POINTS_IN_48H
#     X_chunks = np.array_split(X, num_chunks)
    
#     # Split y into chunks
#     y_chunks = np.array_split(y, num_chunks)
    
#     # Filter out chunks without sleep periods
#     X_chunks_filtered = [X_chunk for X_chunk, y_chunk in zip(X_chunks, y_chunks) if np.sum(y_chunk) != len(y_chunk)]
#     y_chunks_filtered = [y_chunk for y_chunk in y_chunks if np.sum(y_chunk) != len(y_chunk)]
    
#     # Convert to numpy arrays
#     X_array = np.array(X_chunks_filtered)
#     y_array = np.array(y_chunks_filtered)
    
#     return X_array, y_array

# # Now, X_chunked will have a shape like (num_chunks, 48h window, 121) and y_chunked will have a shape like (num_chunks, 48h window, 1)


In [ ]:
def split_and_pad_into_chunks(X, y, chunk_size):
    # Calculate the number of chunks we can form and the size of the last chunk
    num_full_chunks = len(X) // chunk_size
    last_chunk_size = len(X) % chunk_size
    
    # Split the data into full chunks and the last smaller chunk
    X_full_chunks = X[:num_full_chunks * chunk_size].reshape(num_full_chunks, chunk_size, -1)
    y_full_chunks = y[:num_full_chunks * chunk_size].reshape(num_full_chunks, chunk_size, -1)
    
    X_last_chunk = X[num_full_chunks * chunk_size:]
    y_last_chunk = y[num_full_chunks * chunk_size:]
    
    # Pad the last chunk if it's smaller than the required chunk size
    if last_chunk_size > 0:
        padding_size = chunk_size - last_chunk_size
        X_last_chunk = np.pad(X_last_chunk, ((0, padding_size), (0, 0)), mode='constant')
        y_last_chunk = np.pad(y_last_chunk, ((0, padding_size), (0, 0)), mode='constant')
        
        # Add the padded last chunk to our list of chunks
        X_full_chunks = np.vstack((X_full_chunks, X_last_chunk.reshape(1, chunk_size, -1)))
        y_full_chunks = np.vstack((y_full_chunks, y_last_chunk.reshape(1, chunk_size, -1)))
    
    # Filter out chunks without any sleep periods
    valid_indices = [i for i, y_chunk in enumerate(y_full_chunks) if np.sum(y_chunk) != len(y_chunk)]
    X_chunks_filtered = X_full_chunks[valid_indices]
    y_chunks_filtered = y_full_chunks[valid_indices]
    
    return X_chunks_filtered, y_chunks_filtered

# Model

In [ ]:
def dnn_model():
    
#     x_input = Input(shape=(train.shape[-2:]))
    
    x_input = Input(shape=(POINTS_IN_48H, 121))
    
#     layers = [1024, 512, 384, 256, 128]
#     layers = [500, 400, 300, 200, 100]
    
    layers = [256, 128, 64]
    
    x1 = Bidirectional(LSTM(units=layers[0], return_sequences=True))(x_input)
    x2 = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x1)
    x3 = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x2)
#     x4 = Bidirectional(LSTM(units=layers[3], return_sequences=True))(x3)
#     x5 = Bidirectional(LSTM(units=layers[4], return_sequences=True))(x4)
    
#     z2 = Bidirectional(GRU(units=layers[2], return_sequences=True))(x2)
    
#     z31 = Multiply()([x3, z2])
#     z31 = BatchNormalization()(z31)
#     z3 = Bidirectional(GRU(units=layers[3], return_sequences=True))(z31)
    
#     z41 = Multiply()([x4, z3])
#     z41 = BatchNormalization()(z41)
#     z4 = Bidirectional(GRU(units=layers[4], return_sequences=True))(z41)
    
#     z51 = Multiply()([x5, z4])
#     z51 = BatchNormalization()(z51)
#     z5 = Bidirectional(GRU(units=64, return_sequences=True))(z51)
    
#     x = Concatenate(axis=2)([x5, z2, z3, z4, z5])
    
    x = Dense(units=128, activation='selu')(x3)
    
    x_output = Dense(units=1)(x)

    model = Model(inputs=x_input, outputs=x_output, 
                  name='DNN_Model')
    return model

In [ ]:
# model = dnn_model()
# model.summary()

# Configs

In [ ]:
# Parameters
NUM_FOLDS = 7
NUM_EPOCHS = 5
BATCH_SIZE = 8
# GRADIENT_ACCUMULATION_STEPS = 2 

WARMUP_EPOCHS = 1
WARMUP_LR = 1e-5

WEIGHT_DECAY = 1e-4
INIT_LEARNING_RATE = 1e-4
FINAL_LEARNING_RATE = 1e-6

PATIENCE = 3  # For ReduceLROnPlateau
FACTOR = 0.8  # Reduce learning rate by this factor

LSTM_SEQ_LEN = 128
NUM_FEATS = 121


# Model training

In [ ]:
# https://www.kaggle.com/code/illidan7/gbvpp-illifictionlast#Keras-DNN-Model

# https://www.kaggle.com/code/carlmcbrideellis/zzzs-make-small-starter-datasets-target?scriptVersionId=143680507&cellId=15

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

In [ ]:
free_memory()

In [ ]:
# Define the number of data points in a 48-hour chunk
POINTS_IN_48H = int(48 * 60 * 60 / 5)

# Loop over folds
for fold in range(1, 8):
    
    print(f"Fold{fold}")
    
    #########################
    # Collect series ids
    #########################
    
    print("Collect series ids")
    
    validation_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{fold}/fold{fold}/"
    validation_series_ids = [file.split('.pkl')[0] for file in os.listdir(validation_fold_path)]
    
    training_series_ids = []
    for train_fold in range(1, 8):
        if train_fold != fold:
            train_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{train_fold}/fold{train_fold}/"
            training_series_ids.extend([file.split('.pkl')[0] for file in os.listdir(train_fold_path)])
    
    print(f"Num series train: {len(training_series_ids)}")
    print(f"Num series valid: {len(validation_series_ids)}")
    
    
    #########################
    # Model Initialization
    #########################
    
    print("Model Initialization")
    
    # Build the model
    model = dnn_model()

    # Compile the model with custom learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=INIT_LEARNING_RATE)
    model.compile(optimizer=optimizer, loss='mse')
    
    scaler = joblib.load(f"/kaggle/input/zzzs-stdscalerfit/scaler_fold{fold}.pkl")
    
    free_memory()
    
        
    
    #########################
    # Model Training
    #########################
    
    print("Model Training")
    
    for epoch in range(NUM_EPOCHS):
        
        print(f"Epoch {epoch+1}")
        
        # Compute the new learning rate
        new_lr = compute_new_lr(epoch, 
                                initial_lr=INIT_LEARNING_RATE, 
                                final_lr=FINAL_LEARNING_RATE, 
                                total_epochs=NUM_EPOCHS)
        tf.keras.backend.set_value(model.optimizer.lr, new_lr)
        
        # Log learning rate
        # wandb.log({f"lr_{fold}": new_lr})
    
        # Training phase: Train on series from all other folds
        train_losses = []
        for train_fold in tqdm(range(1, 8)):
            if train_fold == fold:  # Skip the current validation fold
                continue
            
            train_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{train_fold}/fold{train_fold}/"
            train_fold_losses = []
            for file in tqdm(os.listdir(train_fold_path)):

                # Load the specific series data
                with open(train_fold_path + file, 'rb') as f:

                    X, y, series_id = joblib.load(f)
                    free_memory()
                    
                    print("Pickle file loaded")
                    
                    # Split the data into chunks and filter
                    X, y = split_and_pad_into_chunks(X, y, POINTS_IN_48H)
                    free_memory()
                    
                    print("Chunks generated")
                                
                    print(f"Series {file.split('.pkl')[0]} training")
                    
                    # Train the model
                    history = model.fit(X, y, epochs=1, batch_size=BATCH_SIZE, verbose=1)
                    free_memory()
                    
                    print(f"Series {file.split('.pkl')[0]} training completed")

                    # Log train loss
                    # wandb.log({f"train_loss_series": history.history['loss'][-1]})
                    
                    train_fold_losses.append(history.history['loss'][-1])

                    # Free up memory
                    del X, y, targets
                    tf.keras.backend.clear_session()
                    free_memory()
                
            train_losses.append(np.mean(train_fold_losses))
        
        # Log the average validation loss for this fold to wandb
        # wandb.log({f"train_loss_fold_{fold}": np.mean(val_losses)})

        # Validation phase: Validate on series from the current fold
        val_losses = []
        for file in os.listdir(validation_fold_path):
            with open(validation_fold_path + file, 'rb') as f:

                X, y, series_id = joblib.load(f)

                # Evaluate the model
                val_loss = model.evaluate(X, y, batch_size=BATCH_SIZE, verbose=0)
                val_losses.append(val_loss)

                del X, y, data, targets
                tf.keras.backend.clear_session()
                free_memory()

        # Log the average validation loss for this fold to wandb
        # wandb.log({f"val_loss_fold_{fold}": np.mean(val_losses)})
    
    # Save the trained model
    model.save(f"./model_fold_{fold}.h5")
    
    
    
    
    break
    

In [ ]:
X.shape

In [ ]:
POINTS_IN_48H

In [ ]:
X, y, series_id = joblib.load('/kaggle/input/zzzs-numpyfolds-fold-1/fold1/3be2f86c3e45.pkl')

print(X.shape)
print(y.shape)
print(series_id)

# Archive

In [ ]:
# # Initialize the scaler outside the training loop
# scaler = RobustScaler()

# # Loop over folds
# for fold in range(1, NUM_FOLDS+1):
    
#     # Directory setup
#     train_fold_path = f'./fold{fold}/'
    
#     # Build the model
#     model = build_lstm_model(input_shape=(LSTM_SEQ_LEN, NUM_FEATS), weight_decay=WEIGHT_DECAY)

#     # Learning rate scheduler
#     lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2)
    
#     # Gradient accumulation setup
#     accumulated_gradients = [tf.zeros_like(var) for var in model.trainable_variables]
#     accumulation_counter = 0
    
#     # Training loop
#     for epoch in range(NUM_EPOCHS):
        
#         # Warmup learning rate adjustment
#         if epoch < WARMUP_EPOCHS:
#             new_lr = LEARNING_RATE * (epoch + 1) / WARMUP_EPOCHS
#             tf.keras.backend.set_value(model.optimizer.lr, new_lr)
        
#         train_losses = []
#         for file in os.listdir(train_fold_path):
            
#             # Data loading
#             targets, data, series_id = joblib.load(train_fold_path + file)
            
#             # Data preprocessing
#             data_np = data.values
#             data_tensor = tf.convert_to_tensor(data_np, dtype=tf.float32)
            
#             # Gradient accumulation
#             with tf.GradientTape() as tape:
#                 predictions = model(data_tensor)
#                 loss = tf.keras.losses.MSE(targets, predictions)
#             grads = tape.gradient(loss, model.trainable_variables)
#             accumulated_gradients = [acc + grad for acc, grad in zip(accumulated_gradients, grads)]
#             accumulation_counter += 1
            
#             if accumulation_counter == GRADIENT_ACCUMULATION_STEPS:
#                 model.optimizer.apply_gradients(zip(accumulated_gradients, model.trainable_variables))
#                 accumulated_gradients = [tf.zeros_like(var) for var in model.trainable_variables]
#                 accumulation_counter = 0
#             train_losses.append(loss)
            
#         # Validation phase
#         val_losses = []
#         for val_fold in range(1, 8):
#             if val_fold != fold:
#                 validation_fold_path = f'./fold{val_fold}/'
#                 for file in os.listdir(validation_fold_path):
#                     # Data loading
#                     targets, data, series_id = joblib.load(validation_fold_path + file)
                    
#                     # Data preprocessing
#                     data_np = data.values
#                     data_tensor = tf.convert_to_tensor(data_np, dtype=tf.float32)
                    
#                     predictions = model(data_tensor)
#                     val_loss = tf.keras.losses.MSE(targets, predictions)
#                     val_losses.append(val_loss)
            
#         # Logging to wandb
#         wandb.log({
#             'epoch': epoch,
#             'train_loss': np.mean(train_losses),
#             'val_loss': np.mean(val_losses),
#             'learning_rate': tf.keras.backend.get_value(model.optimizer.lr)
#         })
        
#         # Apply learning rate scheduler
#         lr_scheduler.on_epoch_end(epoch, {'val_loss': np.mean(val_losses)})
    
#     # Save the trained model
#     model.save(f"./model_fold_{fold}.h5")
